# Bayesian spectral deconvolution with exchange Monte Carlo

This notebook is a guided, beginner-friendly implementation of the method in **Nagata, Sugita & Okada (2012)**, *Bayesian spectral deconvolution with the exchange Monte Carlo method*.

The paper's goal is to decompose a spectrum into Gaussian bands, decide **how many bands** are justified, and avoid getting trapped in a bad local optimum. It does this by combining:

1. a Gaussian radial-basis-function model;
2. a Bayesian posterior distribution;
3. Monte Carlo sampling;
4. exchange Monte Carlo (parallel tempering) to move between temperatures;
5. a marginal-likelihood calculation for model selection.

The implementation below is intentionally verbose. The original paper used substantially longer runs than the quick demonstration here.

## 1. The central problem

Suppose the observed curve has several overlapping peaks. We model it as a sum of K Gaussian-shaped components. For each component k we want its amplitude (strength), center, and width/precision.

The paper calls this a radial basis function (RBF) network. The model is

$$
f(x;\theta)=\sum_{k=1}^K a_k\exp\left[-\frac{b_k}{2}(x-\mu_k)^2\right].
$$

The important distinction is that the paper is **not primarily interested in prediction**. It wants the hidden scientific structure: the number of bands and their parameters.

## 2. Why ordinary optimization is not enough

There are two difficulties. First, the loss surface has local minima because the model is nonlinear and hierarchical. Second, K itself is unknown.

A local optimizer can find a plausible-looking decomposition but miss a better one. Bayesian model selection handles K by comparing the models through their **marginal likelihood** (evidence). Exchange Monte Carlo handles the local-minimum problem by allowing replicas at different temperatures to exchange states.

In [1]:
# The full implementation lives in this Python file.
# Keeping it in a .py file makes it easy to reuse outside the notebook.
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))

from src.bayspecdec import *
import numpy as np

## 3. A synthetic example from the paper

The paper generates 301 x-values from 0 to 3, uses noise variance sigma^2 = 0.01, and uses three Gaussian components with parameters

- amplitudes: 0.587, 1.522, 1.183
- centers: 1.210, 1.455, 1.703
- b parameters: 95.689, 146.837, 164.469.

This gives us a controlled problem where we know the correct answer.

In [2]:
x, y, y_true = make_paper_like_synthetic_data(seed=7)

print('n =', len(x))
print('noise variance =', 0.01)
print('true K = 3')

# plot_synthetic_data(x, y, y_true, Path('notebook_synthetic_data.png'))


n = 301
noise variance = 0.01
true K = 3


## 4. The Bayesian idea in one paragraph

Bayes' rule says

$$
 p(\theta\mid D) \propto p(D\mid\theta)p(\theta).
$$

The likelihood says which parameters explain the data well. The prior says which parameters were plausible before seeing the data. The posterior combines them.

For Gaussian observation noise, the paper writes the posterior in the form

$$
q(\theta;\beta) \propto \exp\left[-\frac{n}{\sigma^2}\beta E(\theta)\right]\phi(\theta),
$$

where E(theta) is the mean-squared error and beta is an artificial inverse-temperature. At beta=1 this is the ordinary posterior (up to normalization); at beta=0 only the prior remains.

## 5. Why introduce temperature?

Imagine a posterior with two separated peaks. At beta=1 the chain sees a complicated landscape and may get stuck in one peak. At beta close to zero the landscape is much flatter and the chain can wander between regions.

Exchange Monte Carlo keeps several chains running simultaneously, at beta values from near zero to 1. Every so often it proposes to **swap the states of adjacent temperatures**. This lets a state at beta=1 effectively travel through a hot chain and come back somewhere else.

In [3]:
beta = beta_schedule(24)
print('first betas:', beta[:6])
print('last betas :', beta[-6:])
# plot_temperature_ladder(beta, Path('notebook_temperature_ladder.png'))


first betas: [0.         0.00013366 0.00020049 0.00030073 0.00045109 0.00067664]
last betas : [0.13168724 0.19753086 0.2962963  0.44444444 0.66666667 1.        ]


## 6. Ordinary Metropolis, step by step

For one replica at fixed beta:

1. Start from a parameter vector theta.
2. Propose theta' = theta + Gaussian random noise.
3. Evaluate the log target at theta'.
4. Accept the proposal with probability

$$
\min\{1,\;q(\theta';\beta)/q(\theta;\beta)\}.
$$

With a symmetric random-walk proposal, the proposal density cancels in the Metropolis-Hastings ratio. Working in log space gives

$$
\log\alpha = \log q(\theta';\beta)-\log q(\theta;\beta).
$$

This is why the implementation can avoid ever computing the normalized posterior density.

## 7. Exchange move, step by step

Take adjacent replicas l and l+1. They currently contain theta_l and theta_(l+1). We propose to swap them. The paper derives

$$
\alpha_{swap}=\min\left(1,\exp\left[\frac{n}{\sigma^2}(\beta_{l+1}-\beta_l)\{E(\theta_{l+1})-E(\theta_l)\}\right]\right).
$$

Interpretation:

- $\beta_{l+1} > \beta_l$, so replica $l+1$ is colder.
- A low-energy state is preferred by the colder replica.
- If a hot replica currently has a particularly good state, the swap is easy to accept.
- Repeated swaps let good states move back and forth through temperature space.

In [4]:
# A tiny direct check of the swap exponent.
problem = BayesianRBFProblem(x=x, y=y, sigma2=0.01, K=3, prior=SyntheticPrior())

beta1, beta2 = 0.2, 0.8
E_low, E_high = 0.1, 1.0
log_v = (len(x) / problem.sigma2) * (beta2 - beta1) * (E_high - E_low)
print('log swap-ratio when colder state is worse:', log_v)
print('=> the proposed exchange is strongly disfavored, as expected.')


log swap-ratio when colder state is worse: 16254.000000000004
=> the proposed exchange is strongly disfavored, as expected.


## 8. The most important Bayesian calculation: evidence

For model size K the paper defines

$$
Z(\beta)=\int \exp\left[-\frac{n}{\sigma^2}\beta E(\theta)\right]\phi(\theta)\,d\theta.
$$

At beta=0, Z(0)=1. At beta=1, Z(1) is the marginal likelihood/evidence.

Instead of evaluating the difficult integral directly, the paper breaks it into ratios:

$$
Z(1)=\prod_l \frac{Z(\beta_{l+1})}{Z(\beta_l)}.
$$

Each ratio can be written as an expectation under q(theta; beta_l), which is exactly the kind of expectation that MCMC supplies.

This is the key bridge: **exchange MC gives us samples at many beta values, and those samples let us estimate the evidence.**

## 9. Running exchange Monte Carlo

The original synthetic experiment used 24 replicas, 10,000 burn-in steps, and 10,000 expectation steps. The demonstration below deliberately uses much shorter chains so it runs quickly on a laptop. Longer chains generally give more reliable evidence estimates and posterior summaries.

In [5]:
prior = SyntheticPrior()
problem3 = BayesianRBFProblem(x, y, sigma2=0.01, K=3, prior=prior)

rng = np.random.default_rng(123)
emc = ExchangeMonteCarlo(problem3, beta=beta_schedule(12), rng=rng)
result3 = emc.run(burn_in=600, expectation_steps=600)

evidence3 = estimate_log_evidence_from_exchange(problem3, result3)
print('stochastic complexity -log Z =', -evidence3.log_z)
print('beta=1 within-chain acceptance =', result3.within_acceptance[-1])
print('mean adjacent-swap acceptance =', result3.exchange_acceptance.mean())


stochastic complexity -log Z = 171.2390589079083
beta=1 within-chain acceptance = 0.0275
mean adjacent-swap acceptance = 0.36333333333333334


## 10. Inspect the chains

A good habit in Monte Carlo work is to inspect diagnostics rather than trusting a single final number. Here we look at energy traces for a hot, middle, and cold replica.

The hot chain should explore more broadly; the cold chain should concentrate around low-error regions. Exchange moves are what connect their exploration.

In [6]:
run3 = ModelRun(K=3, problem=problem3, exchange_result=result3, evidence=evidence3)
# plot_exchange_diagnostics(run3, Path('notebook_energy_traces.png'))
print('Acceptance by temperature:')
for i, (b, a) in enumerate(zip(result3.beta, result3.within_acceptance)):
    print(f'  replica {i+1:2d}, beta={b: .4f}, acceptance={a: .3f}')


Acceptance by temperature:
  replica  1, beta= 0.0000, acceptance= 0.912
  replica  2, beta= 0.0173, acceptance= 0.613
  replica  3, beta= 0.0260, acceptance= 0.532
  replica  4, beta= 0.0390, acceptance= 0.438
  replica  5, beta= 0.0585, acceptance= 0.352
  replica  6, beta= 0.0878, acceptance= 0.244
  replica  7, beta= 0.1317, acceptance= 0.182
  replica  8, beta= 0.1975, acceptance= 0.140
  replica  9, beta= 0.2963, acceptance= 0.074
  replica 10, beta= 0.4444, acceptance= 0.050
  replica 11, beta= 0.6667, acceptance= 0.035
  replica 12, beta= 1.0000, acceptance= 0.028


## 11. Model selection

For each candidate K, we run the same Bayesian calculation and estimate $-\log{\mathrm{Z}}$. Because the paper uses a uniform prior on K in its synthetic experiment, choosing the model with the largest evidence is equivalent to choosing the model with the smallest stochastic complexity $-\log{\mathrm{Z}}$.

A lower value is better.

In [7]:
runs = run_model_selection(
    x=x,
    y=y,
    sigma2=0.01,
    K_values=[1, 2, 3],
    prior=prior,
    L=24,
    burn_in=2000,
    expectation_steps=10000,
    seed=123,
)

# plot_model_selection(runs, Path('notebook_model_selection.png'))
best = min(runs, key=lambda r: r.stochastic_complexity)
print(f" Selected K = {best.K} ")


K= 1 | stochastic complexity -log Z =  319.205 | beta=1 MH acc=0.069 | mean swap acc=0.754
K= 2 | stochastic complexity -log Z =  235.898 | beta=1 MH acc=0.031 | mean swap acc=0.706
K= 3 | stochastic complexity -log Z =  156.100 | beta=1 MH acc=0.010 | mean swap acc=0.692
 Selected K = 3 


## 12. Recovering the bands

Once K is chosen, the paper estimates the posterior mode parameters. In this educational implementation we approximate that by taking the sampled $\beta=1$ point with the highest log posterior. Because Gaussian components are exchangeable, the order of the components is arbitrary: the same fit can appear with the bands in a different order.

In [8]:
theta_hat = best.posterior_mode_sample
K = best.K
print('a  =', theta_hat[:K])
print('mu =', theta_hat[K:2*K])
print('b  =', theta_hat[2*K:])

# plot_fit(best, x, y, Path('notebook_selected_fit.png'))
# plot_parameter_histograms(best, Path('notebook_parameter_histograms.png'))


a  = [1.49427685 0.54791652 1.1775474 ]
mu = [1.45261548 1.20251478 1.70232167]
b  = [146.90638585  92.18167614 153.61373102]
